<a href="https://colab.research.google.com/github/susanavenda/data_cambridge/blob/main/CAM_DS_C301_Hyperparameter_tuning_Demo_2_2_3_Part_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Demonstration 2.2.3 Hyperparameter tuning Part 1

In this demonstration, we will explore hyperparameter tuning with a specific focus on recurrent neural networks (RNNs). While we have already covered hyperparameter tuning for generic deep learning networks, this demonstration will build on those concepts. We will delve into considerations and parameters unique to fine-tuning RNNs, particularly for handling sequence data in natural language processing. We will compare the performance of models using pre-trained word embeddings like Word2Vec and building custom embeddings during the training process using TensorFlow.

In [ ]:
!pip install datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.8/547.8 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 MB 11.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 7.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 7.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 16.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.31.0
    Uninstalling requests-2.31.0:
      Successfully uninstalled requests-2.31.0
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 14.0.2
    Uninstalling pyarrow-14.0.2:
      Successfully uninstalled pyarrow-14.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 24

In [ ]:
from datasets import load_dataset
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from transformers import BertTokenizer, BertModel
from tensorflow.keras.layers import Embedding, SimpleRNN, Dropout, Dense, LSTM, GRU, Bidirectional
import gensim.downloader as api
import pandas as pd
import random

In [ ]:
# Define a function to reset the session.
def reset_session():
    tf.keras.backend.clear_session()
    np.random.seed(42)
    random.seed(42)
    tf.random.set_seed(42)

In [ ]:
dataset = load_dataset("imdb")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [ ]:
#Create dataframes of the train and validation split.
text_train = dataset['train']['text']
label_train = dataset['train']['label']
text_test = dataset['test']['text']
label_test = dataset['test']['label']

In [ ]:
df_train = pd.DataFrame()
df_train['text'] = text_train
df_train['label'] = label_train

In [ ]:
df_train

,text,label
0,I rented I AM CURIOUS-YELLOW from my video sto...,0
1,"""I Am Curious: Yellow"" is a risible and preten...",0
2,If only to avoid making this type of film in t...,0
3,This film was probably inspired by Godard's Ma...,0
4,"Oh, brother...after hearing about this ridicul...",0
...,...,...
24995,A hit at the time but now better categorised a...,1
24996,I love this movie like no other. Another time ...,1
24997,This film and it's sequel Barry Mckenzie holds...,1
24998,'The Adventures Of Barry McKenzie' started lif...,1


In [ ]:
from sklearn.model_selection import train_test_split
train, valid = train_test_split(df_train,  test_size=0.2)

In [ ]:
text_train =  train['text'].tolist()
label_train = train['label'].tolist()
text_valid = valid['text'].tolist()
label_valid = valid['label'].tolist()

In [ ]:
# Parameters
vocab_size = 10000
max_length = 150
padding_type = 'post'
trunc_type = 'post'

# Initialise the tokeniser.
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(dataset['train']['text'])

# Tokenise and pad sequences.
train_sequences = tokenizer.texts_to_sequences(text_train)
train_padded = pad_sequences(train_sequences, maxlen=max_length, padding=padding_type, truncating=trunc_type)

valid_sequences = tokenizer.texts_to_sequences(text_valid)
valid_padded = pad_sequences(valid_sequences, maxlen=max_length, padding=padding_type, truncating=trunc_type)

test_sequences = tokenizer.texts_to_sequences(text_test)
test_padded = pad_sequences(test_sequences, maxlen=max_length, padding=padding_type, truncating=trunc_type)

# Prepare labels
train_labels = np.array(label_train)
valid_labels = np.array(label_valid)
test_labels = np.array(label_test)

In [ ]:
# Load Word2Vec embeddings from Gensim.
word2vec_model = api.load("word2vec-google-news-300")

# Initialise embedding matrix.
embedding_dim = 300  # Dimension of Word2Vec embeddings
embedding_matrix = np.zeros((vocab_size, embedding_dim))


for word, i in tokenizer.word_index.items():
    if i < vocab_size:
        try:
            embedding_vector = word2vec_model[word]
            embedding_matrix[i] = embedding_vector
        except KeyError:
            pass  # Word not found in the pre-trained model

[==================================================] 100.0% 1662.8/1662.8MB downloaded


### Model 1: RNN(128); Word2Vec


In [ ]:
reset_session()
# Define the model.
model1 = Sequential([
    # Embedding layer using the pre-trained embedding matrix
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, weights=[embedding_matrix], input_length=max_length, trainable=False),

    # Simple RNN layer
    SimpleRNN(128, return_sequences=False),

    # Dense layer with sigmoid activation for binary classification
    Dense(1, activation='sigmoid')
])

# Compile the model.
model1.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# Display the model summary.
model1.summary()

model1.fit(train_padded, train_labels, validation_data=(valid_padded, valid_labels), epochs=8, batch_size=64)

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 150, 300)          3000000   
                                                                 
 simple_rnn (SimpleRNN)      (None, 128)               54912     
                                                                 
 dense (Dense)               (None, 1)                 129       
                                                                 
Total params: 3055041 (11.65 MB)
Trainable params: 55041 (215.00 KB)
Non-trainable params: 3000000 (11.44 MB)
_________________________________________________________________
Epoch 1/8
313/313 [==============================] - 51s 156ms/step - loss: 0.6980 - accuracy: 0.5184 - val_loss: 0.6956 - val_accuracy: 0.5008
Epoch 2/8
313/313 [==============================] - 48s 155ms/step - loss: 0.6905 - accuracy: 0.5265 - val_loss: 0.6920 - val_accur

In [ ]:
# Final evaluation of the model
scores = model1.evaluate(test_padded, test_labels, verbose=0)
print("Accuracy: %.2f%%" % (scores[1]*100))

Accuracy: 54.73%


### Model 2: RNN(50); Word2Vec

In [ ]:
reset_session()
# Define the model.
model2 = Sequential([
    # Embedding layer using the pre-trained embedding matrix
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, weights=[embedding_matrix], input_length=max_length, trainable=False),

    # Simple RNN layer
    SimpleRNN(50, return_sequences=False),

    # Dense layer with sigmoid activation for binary classification
    Dense(1, activation='sigmoid')
])

# Compile the model
model2.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# Display the model summary
model2.summary()

model2.fit(train_padded, train_labels, validation_data=(valid_padded, valid_labels), epochs=8, batch_size=64)

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 150, 300)          3000000   
                                                                 
 simple_rnn (SimpleRNN)      (None, 50)                17550     
                                                                 
 dense (Dense)               (None, 1)                 51        
                                                                 
Total params: 3017601 (11.51 MB)
Trainable params: 17601 (68.75 KB)
Non-trainable params: 3000000 (11.44 MB)
_________________________________________________________________
Epoch 1/8
313/313 [==============================] - 20s 58ms/step - loss: 0.6949 - accuracy: 0.5283 - val_loss: 0.6929 - val_accuracy: 0.5140
Epoch 2/8
313/313 [==============================] - 18s 58ms/step - loss: 0.6847 - accuracy: 0.5392 - val_loss: 0.6874 - val_accuracy

In [ ]:
# Final evaluation of the model
scores = model2.evaluate(test_padded, test_labels, verbose=0)
print("Accuracy: %.2f%%" % (scores[1]*100))

Accuracy: 59.50%


### Model 3: RNN(50); train embedding vector with vector length = 30


In [ ]:
reset_session()
embedding_vector_length = 30
model3 = Sequential([
    # Embedding vectors created during the training cycle
    Embedding(vocab_size, embedding_vector_length, input_length=max_length),

    # Simple RNN layer
    SimpleRNN(50, return_sequences=False),

    # Dense layer with sigmoid activation for binary classification
    Dense(1, activation='sigmoid')
])

# Compile the model.
model3.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# Display the model summary.
model3.summary()

model3.fit(train_padded, train_labels, validation_data=(valid_padded, valid_labels), epochs=5, batch_size=64)

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 150, 30)           300000    
                                                                 
 simple_rnn (SimpleRNN)      (None, 50)                4050      
                                                                 
 dense (Dense)               (None, 1)                 51        
                                                                 
Total params: 304101 (1.16 MB)
Trainable params: 304101 (1.16 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
Epoch 1/5
313/313 [==============================] - 21s 58ms/step - loss: 0.6942 - accuracy: 0.5076 - val_loss: 0.6945 - val_accuracy: 0.5110
Epoch 2/5
313/313 [==============================] - 23s 74ms/step - loss: 0.6508 - accuracy: 0.6162 - val_loss: 0.7216 - val_accuracy: 0.522

In [ ]:
# Final evaluation of the model
scores = model3.evaluate(test_padded, test_labels, verbose=0)
print("Accuracy: %.2f%%" % (scores[1]*100))

Accuracy: 51.09%


### Model 4: RNN(50); train embedding vector with vector length = 300

In [ ]:
reset_session()
embedding_vector_length = 300
model4 = Sequential([
    # Embedding vectors created during the training cycle
    Embedding(vocab_size, embedding_vector_length, input_length=max_length),

    # Simple RNN layer
    SimpleRNN(50, return_sequences=False),

    # Dense layer with sigmoid activation for binary classification
    Dense(1, activation='sigmoid')
])

# Compile the model.
model4.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# Display the model summary.
model4.summary()

model4.fit(train_padded, train_labels, validation_data=(valid_padded, valid_labels), epochs=5, batch_size=64)

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 150, 300)          3000000   
                                                                 
 simple_rnn (SimpleRNN)      (None, 50)                17550     
                                                                 
 dense (Dense)               (None, 1)                 51        
                                                                 
Total params: 3017601 (11.51 MB)
Trainable params: 3017601 (11.51 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
Epoch 1/5
313/313 [==============================] - 40s 125ms/step - loss: 0.6843 - accuracy: 0.5683 - val_loss: 0.6701 - val_accuracy: 0.6082
Epoch 2/5
313/313 [==============================] - 37s 118ms/step - loss: 0.6020 - accuracy: 0.6602 - val_loss: 0.7373 - val_accuracy:

In [ ]:
# Final evaluation of the model
scores = model4.evaluate(test_padded, test_labels, verbose=0)
print("Accuracy: %.2f%%" % (scores[1]*100))

Accuracy: 51.40%


### Model 5: RNN(128): train embedding layer with vector length = 300


In [ ]:
reset_session()
embedding_vector_length = 300
model5 = Sequential([
    # Embedding vectors created during the training cycle
    Embedding(vocab_size, embedding_vector_length, input_length=max_length),

    # Simple RNN layer
    SimpleRNN(128, return_sequences=False),

    # Dense layer with sigmoid activation for binary classification
    Dense(1, activation='sigmoid')
])

# Compile the model.
model5.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# Display the model summary.
model5.summary()

model5.fit(train_padded, train_labels, validation_data=(valid_padded, valid_labels), epochs=5, batch_size=64)

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 150, 300)          3000000   
                                                                 
 simple_rnn (SimpleRNN)      (None, 128)               54912     
                                                                 
 dense (Dense)               (None, 1)                 129       
                                                                 
Total params: 3055041 (11.65 MB)
Trainable params: 3055041 (11.65 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
Epoch 1/5
313/313 [==============================] - 71s 222ms/step - loss: 0.6977 - accuracy: 0.5107 - val_loss: 0.6927 - val_accuracy: 0.5164
Epoch 2/5
313/313 [==============================] - 65s 207ms/step - loss: 0.6187 - accuracy: 0.6362 - val_loss: 0.7553 - val_accuracy:

In [ ]:
# Final evaluation of the model
scores = model5.evaluate(test_padded, test_labels, verbose=0)
print("Accuracy: %.2f%%" % (scores[1]*100))

Accuracy: 51.29%


### Model 6: RNN(128); train embedding layer with vector length = 30

In [ ]:
reset_session()
embedding_vector_length = 30
model6 = Sequential([
    # Embedding vectors created during the training cycle
    Embedding(vocab_size, embedding_vector_length, input_length=max_length),

    # Simple RNN layer
    SimpleRNN(128, return_sequences=False),

    # Dense layer with sigmoid activation for binary classification
    Dense(1, activation='sigmoid')
])

# Compile the model.
model6.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# Display the model summary.
model6.summary()

model6.fit(train_padded, train_labels, validation_data=(valid_padded, valid_labels), epochs=5, batch_size=64)

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 150, 30)           300000    
                                                                 
 simple_rnn (SimpleRNN)      (None, 128)               20352     
                                                                 
 dense (Dense)               (None, 1)                 129       
                                                                 
Total params: 320481 (1.22 MB)
Trainable params: 320481 (1.22 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
Epoch 1/5
313/313 [==============================] - 27s 79ms/step - loss: 0.6982 - accuracy: 0.5001 - val_loss: 0.6949 - val_accuracy: 0.5054
Epoch 2/5
313/313 [==============================] - 24s 76ms/step - loss: 0.6968 - accuracy: 0.5117 - val_loss: 0.7029 - val_accuracy: 0.495

In [ ]:
# Final evaluation of the model
scores = model6.evaluate(test_padded, test_labels, verbose=0)
print("Accuracy: %.2f%%" % (scores[1]*100))

Accuracy: 49.90%


### Model 7: RNN(128); train embedding layer with vector length = 300

In [ ]:
reset_session()
embedding_vector_length = 300
model7 = Sequential([
    # Embedding vectors created during the training cycle
    Embedding(vocab_size, embedding_vector_length, input_length=max_length),

    # Simple RNN layer
    SimpleRNN(128, return_sequences=False, recurrent_dropout = 0.5),

    # Dense layer with sigmoid activation for binary classification
    Dense(1, activation='sigmoid')
])

# Compile the model.
model7.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# Display the model summary.
model7.summary()

model7.fit(train_padded, train_labels, validation_data=(valid_padded, valid_labels), epochs=5, batch_size=64)

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 150, 300)          3000000   
                                                                 
 simple_rnn (SimpleRNN)      (None, 128)               54912     
                                                                 
 dense (Dense)               (None, 1)                 129       
                                                                 
Total params: 3055041 (11.65 MB)
Trainable params: 3055041 (11.65 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
Epoch 1/5
313/313 [==============================] - 99s 308ms/step - loss: 0.7406 - accuracy: 0.4981 - val_loss: 0.6987 - val_accuracy: 0.4916
Epoch 2/5
313/313 [==============================] - 94s 300ms/step - loss: 0.7078 - accuracy: 0.5020 - val_loss: 0.6934 - val_accuracy:

In [ ]:
# Final evaluation of the model
scores = model7.evaluate(test_padded, test_labels, verbose=0)
print("Accuracy: %.2f%%" % (scores[1]*100))

Accuracy: 50.25%


## Key information
In this demonstration, we explored hyperparameter tuning for recurrent neural networks (RNNs), including the use of pre-trained word embeddings and custom embeddings. Various models were tested to address overfitting and improve performance, but challenges remained with validation and test accuracy.

## Reflect
What are the practical applications of these techniques?
> Select the pen from the toolbar to add your entry.